# 03 — Prepare & Export

Merge the cleaned `irs_edges` + `acs_edges` staging tables into the unified **`migration_flows`** table — one row per **origin × destination × year**, with IRS and ACS measures side by side. Persist `migration_flows` and `nonmigrants` as first-class DuckDB tables, save processed Parquet, and package the sellable export with a codebook.

**`migration_flows` schema (one directed edge per row):**

| group | columns |
|---|---|
| keys | `year`, `origin_fips`, `origin_state`, `origin_state_name`, `origin_type`, `dest_fips`, `dest_state`, `dest_state_name`, `dest_type` |
| IRS | `irs_returns_out/_in`, `irs_individuals_out/_in`, `irs_agi_out/_in` |
| ACS | `acs_migrants`, `acs_moe`, `acs_moe_pct`, `acs_reliable` |

Read "incoming" vs "outgoing" by filtering: rows where `dest_state = X` are moves **into** X; rows where `origin_state = X` are moves **out of** X.

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd

PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

from src.ingest import load_config
from src.clean_quality import get_connection, load_to_duckdb, run_sql, quality_report, save_processed
from src.prepare import (build_migration_flows, build_nonmigrants, package_dataset,
    build_county_edges, build_county_income_class, build_county_migration_flows)

cfg = load_config('config.yaml')
con = get_connection(cfg)
print(f'Project: {cfg["project_name"]}')

## 1. Merge IRS + ACS into `migration_flows`
Full outer join of `irs_edges` and `acs_edges` on (origin, dest, year), joined to `state_ref` for names/types. Staging tables were created by `02-clean`; if this notebook is run standalone, reload them from interim Parquet first.

In [ ]:
# Ensure staging tables exist (in case 02-clean wasn't run in this session)
have = set(con.execute("SELECT table_name FROM information_schema.tables").df()['table_name'])
interim = Path(cfg['paths']['data_interim'])
for t in ['state_ref', 'irs_edges', 'acs_edges']:
    if t not in have:
        load_to_duckdb(pd.read_parquet(interim / f'{t}.parquet'), t, con)
        print(f'reloaded {t} from interim')

flows = build_migration_flows(con)
print(f'migration_flows: {len(flows):,} rows × {len(flows.columns)} cols')
flows.head()

## 2. Derived reliability columns (ACS)
ACS is a survey, so small flows carry large margins of error. Add:
- `acs_moe_pct` — margin of error as a fraction of the estimate
- `acs_reliable` — `True` when MOE < 50% of the estimate (a common usability cutoff)

Net migration and gross flows are intentionally **not** added here — they are aggregations that belong in `05-analysis`, since they require pairing A→B with B→A.

In [ ]:
load_to_duckdb(flows, '_flows_stage', con)
flows = run_sql("""
    SELECT *,
        CASE WHEN acs_migrants IS NULL OR acs_migrants = 0 THEN NULL
             ELSE ROUND(acs_moe * 1.0 / acs_migrants, 4) END AS acs_moe_pct,
        CASE WHEN acs_migrants IS NULL OR acs_migrants = 0 THEN NULL
             ELSE (acs_moe * 1.0 / acs_migrants) < 0.50 END AS acs_reliable
    FROM _flows_stage
""", con)
con.execute('DROP TABLE IF EXISTS _flows_stage')
print(f'reliable ACS edges: {int(flows.acs_reliable.sum()):,} / {flows.acs_migrants.notna().sum():,} with an estimate')
flows[['year','origin_state','dest_state','acs_migrants','acs_moe','acs_moe_pct','acs_reliable']].head()

## 3. Persist tables to DuckDB
Write `migration_flows` and `nonmigrants` as first-class tables, and register both in `_sources` (using the project's existing schema).

In [ ]:
load_to_duckdb(flows, 'migration_flows', con)

# nonmigrants was built in 02-clean; rebuild here so 03 is self-contained
nonmigrants = build_nonmigrants(con)
load_to_duckdb(nonmigrants, 'nonmigrants', con)

from datetime import date
today = date.today().isoformat()
def register(table, source_name, source_url, description, n):
    con.execute('INSERT OR REPLACE INTO _sources (table_name, source_name, source_url, description, retrieved_date, row_count) VALUES (?, ?, ?, ?, ?, ?)',
                [table, source_name, source_url, description, today, int(n)])

register('migration_flows', 'IRS SOI + Census ACS (merged)', '',
         'Unified directed-edge migration table: one row per origin x destination x year, IRS (out/in) and ACS (migrants/MOE) side by side. True state-to-state edges only.', len(flows))
register('nonmigrants', 'IRS SOI Migration Data', 'https://www.irs.gov/statistics/soi-tax-stats-migration-data',
         'IRS rows where origin_fips==dest_fips (filed in same state both years). One row per state x year.', len(nonmigrants))
print('persisted migration_flows and nonmigrants to DuckDB')

## 4. Quality report

In [ ]:
_ = quality_report(flows, 'migration_flows', con,
                   required_columns=['year','origin_fips','dest_fips','origin_state','dest_state'],
                   max_null_pct=0.60)
print()
# grain check: (origin, dest, year) must be unique
dupes = run_sql("""
    SELECT origin_fips, dest_fips, year, COUNT(*) c
    FROM migration_flows GROUP BY 1,2,3 HAVING COUNT(*) > 1
""", con)
print(f'duplicate (origin,dest,year) keys: {len(dupes)}  (should be 0)')

## 5. Save processed Parquet

In [ ]:
save_processed(flows,       cfg, 'migration_flows.parquet')
save_processed(nonmigrants, cfg, 'nonmigrants.parquet')

## 7. County-to-county flows with income-class tags
Join the county edges to each county's income class (above/below national median AGI-per-return) on both endpoints, producing `county_migration_flows` — one row per origin county × dest county × year with a `flow_class` (e.g. `above_to_below`).

If 02-clean wasn't run this session, the staging tables are rebuilt from `county_raw` first.

In [ ]:
have = set(con.execute('SELECT table_name FROM information_schema.tables').df()['table_name'])
if 'county_edges' not in have:
    load_to_duckdb(build_county_edges(con), 'county_edges', con)
if 'county_income_class' not in have:
    from src.prepare import build_county_nonmigrants
    load_to_duckdb(build_county_nonmigrants(con), 'county_nonmigrants', con)
    load_to_duckdb(build_county_income_class(con), 'county_income_class', con)

county_flows = build_county_migration_flows(con)
load_to_duckdb(county_flows, 'county_migration_flows', con)
print(f'county_migration_flows: {len(county_flows):,} rows')
print(con.execute('SELECT flow_class, COUNT(*) edges, SUM(irs_individuals_out) people FROM county_migration_flows GROUP BY flow_class ORDER BY edges DESC').df().to_string(index=False))

from datetime import date
today = date.today().isoformat()
IRS_URL = 'https://www.irs.gov/statistics/soi-tax-stats-migration-data'
register = lambda t,sn,su,d,n: con.execute('INSERT OR REPLACE INTO _sources (table_name, source_name, source_url, description, retrieved_date, row_count) VALUES (?,?,?,?,?,?)',[t,sn,su,d,today,int(n)])
register('county_migration_flows', 'IRS SOI Migration Data (derived)', IRS_URL,
         'County-to-county directed flows, 2023, with origin/dest income class (above/below national median AGI-per-return) and combined flow_class.', len(county_flows))

save_processed(county_flows, cfg, 'county_migration_flows.parquet')

### Package county export (CSV + Excel + Parquet + codebook)

In [ ]:
county_codebook = {
    'year': 'Migration year = second year of the IRS pair (2023 = the 2022–2023 move).',
    'origin_fips': '5-digit county GEOID people moved FROM (state FIPS + county FIPS).',
    'origin_name': 'Origin county name.',
    'origin_agi_per_return': 'Origin county income proxy: non-migrant AGI per return, in dollars.',
    'origin_income_class': "'above_median' or 'below_median' vs the national median county AGI-per-return.",
    'dest_fips': '5-digit county GEOID people moved TO.',
    'dest_name': 'Destination county name.',
    'dest_agi_per_return': 'Destination county income proxy: non-migrant AGI per return, in dollars.',
    'dest_income_class': "'above_median' or 'below_median' for the destination county.",
    'flow_class': 'Combined origin→dest income class: above_to_above / above_to_below / below_to_above / below_to_below.',
    'irs_returns_out': 'IRS returns (~households) for this county move, from the origin outflow file.',
    'irs_returns_in': 'IRS returns from the destination inflow file.',
    'irs_individuals_out': 'IRS individuals (~people), origin outflow file.',
    'irs_individuals_in': 'IRS individuals, destination inflow file.',
    'irs_agi_out': 'AGI for this move (THOUSANDS of dollars), origin outflow file.',
    'irs_agi_in': 'AGI (THOUSANDS of dollars), destination inflow file.',
}
county_notes = '''
Source: IRS SOI county-to-county migration (per-state Excel files) — https://www.irs.gov/statistics/soi-tax-stats-migration-data (public domain)

Grain: one row per origin county x destination county x year (directed edge). True county-to-county moves only;
summary rows, foreign, and non-migrant self-edges excluded (non-migrants used only to derive the income proxy).

Income class: each county is flagged above/below the NATIONAL MEDIAN of county AGI-per-return, where a county's
AGI-per-return is its non-migrant total AGI / non-migrant returns (a proxy for resident income). The median is
computed across all ~3,140 counties, one vote per county. This is a county-level income proxy, NOT the income of
the people in each specific flow (IRS does not publish migration by income bracket).
'''
written_county = package_dataset(county_flows, cfg, name='county_migration_flows_v1',
                                 codebook=county_codebook, notes=county_notes)
written_county

## 8. County chart-ready tables (phase-2 county story)
These are the shaped / aggregated per-chart tables that the `04-viz` county section
**consumes** — 03 owns the chart data, 04 only reads it. Built from
`county_migration_flows` + `county_income_class` (both already in DuckDB), each is
saved to `data/processed/` as Parquet and loaded back into DuckDB.
Four tables:
1. **`chart_county_net`** — one row per county: people in/out, net people (in − out),
   AGI in/out, net AGI (in − out), tagged by income class, plus a **`label`** column
   (county name + 2-letter state in parens, e.g. `Los Angeles County (CA)`). Source for
   the net gainers/losers chart.
2. **`chart_county_income_class_net`** — one row per income class, the net migration of
   **people** and **dollars**, both on the SAME honest **in − out** convention
   (`SUM(in) − SUM(out)`).
3. **`chart_county_top_pairs`** — top 20 directed county→county pairs by IRS individuals.
4. **`chart_la_destinations`** — Los Angeles County (`06037`) outbound detail: one row per
   destination county, with `people`, `agi_millions`, and `agi_per_person`. Source for the
   LA two-panel detail chart.
**Honest net convention (both measures same direction):** in `chart_county_income_class_net`,
both `net_people` and `net_agi_billions` use the plain **arrivals − departures** reference
(`SUM(in) − SUM(out)`). Under this convention, below-median ("cheaper") counties were net
gainers of BOTH **people** (+59,420) AND **income** (+$1.31B) in 2023 — they did NOT lose
income. (An earlier version stored `net_agi` with the opposite sign — "AGI leaving the class" —
to force a "cheaper counties lost income" story; that was misleading and has been removed. No
table encodes the sign-flipped convention anymore.) `chart_county_net` already used the plain
in − out convention for both measures.
Caveats carried into the charts: **2023 only**; identified county→county edges cover only
~60% of moves; **income class = place, not mover** (IRS does not publish migration by
income bracket).


In [ ]:
# 8a. chart_county_net — one row per county (people/AGI in, out, net), income-class tagged, + state-in-parens label
# `label` = county name + 2-letter state in parens (e.g. 'Los Angeles County (CA)'), derived from
# LEFT(fips,2) -> state_ref.abbrev. state_ref is already present in the DB (built in earlier stages;
# confirmed full coverage of every county prefix). There is no build_state_ref helper in src.prepare,
# so we join the existing state_ref table rather than rebuild it.
chart_county_net = run_sql('''
    WITH o AS (SELECT origin_fips f, SUM(irs_individuals_out) people_out, SUM(irs_agi_out) agi_out
               FROM county_migration_flows GROUP BY 1),
         i AS (SELECT dest_fips f, SUM(irs_individuals_in) people_in, SUM(irs_agi_in) agi_in
               FROM county_migration_flows GROUP BY 1)
    SELECT ic.fips, ic.name, ic.income_class,
           COALESCE(i.people_in,0)  AS people_in,
           COALESCE(o.people_out,0) AS people_out,
           COALESCE(i.people_in,0) - COALESCE(o.people_out,0) AS net_people,
           COALESCE(i.agi_in,0)  AS agi_in,
           COALESCE(o.agi_out,0) AS agi_out,
           COALESCE(i.agi_in,0) - COALESCE(o.agi_out,0) AS net_agi,
           ic.name || ' (' || sr.abbrev || ')' AS label
    FROM county_income_class ic
    LEFT JOIN o ON o.f = ic.fips
    LEFT JOIN i ON i.f = ic.fips
    LEFT JOIN state_ref sr ON sr.fips = LEFT(ic.fips, 2)
''', con)
save_processed(chart_county_net, cfg, 'chart_county_net.parquet')
load_to_duckdb(chart_county_net, 'chart_county_net', con)
register('chart_county_net', 'IRS SOI county migration (prepared)', IRS_URL,
         'One row per county (2023): people_in/out, net_people (in-out), agi_in/out, net_agi (in-out), income_class, '
         'and label (county name + state abbrev in parens, e.g. "Los Angeles County (CA)"). Source for the county net gainers/losers chart.',
         len(chart_county_net))
print(f'chart_county_net: {len(chart_county_net):,} rows')
print(chart_county_net.loc[chart_county_net['name'].isin(['Los Angeles County','Polk County']), ['label','income_class','net_people','net_agi']].to_string(index=False))


In [ ]:
# 8b. chart_county_income_class_net — net people AND net AGI, both on the plain IN - OUT convention
# (arrivals - departures). This replaces an earlier version that stored net_agi with the opposite
# sign ("AGI leaving the class") to manufacture a "cheaper counties lost income" story. Under the
# honest in - out reading, below-median counties gained BOTH people (+59,420) AND income (+$1.31B).
chart_county_income_class_net = run_sql('''
    WITH o AS (SELECT origin_fips f, SUM(irs_individuals_out) go, SUM(irs_agi_out) ao
               FROM county_migration_flows GROUP BY 1),
         i AS (SELECT dest_fips f, SUM(irs_individuals_in) ca, SUM(irs_agi_in) ai
               FROM county_migration_flows GROUP BY 1)
    SELECT ic.income_class,
           SUM(COALESCE(ca,0) - COALESCE(go,0)) AS net_people,
           ROUND(SUM(COALESCE(ai,0) - COALESCE(ao,0)) / 1e6, 3) AS net_agi_billions
    FROM county_income_class ic
    LEFT JOIN o ON o.f = ic.fips
    LEFT JOIN i ON i.f = ic.fips
    GROUP BY 1 ORDER BY income_class
''', con)
save_processed(chart_county_income_class_net, cfg, 'chart_county_income_class_net.parquet')
load_to_duckdb(chart_county_income_class_net, 'chart_county_income_class_net', con)
register('chart_county_income_class_net', 'IRS SOI county migration (prepared)', IRS_URL,
         'Net migration by county income class (2023), both measures on the plain IN - OUT convention: '
         'net_people = SUM(in) - SUM(out); net_agi_billions = (SUM(in) - SUM(out))/1e6 (AGI is in '
         'thousands, /1e6 -> billions). Below-median counties are net gainers of BOTH people (+59,420) '
         'and income (+$1.31B); above-median are net losers of both.',
         len(chart_county_income_class_net))
print(chart_county_income_class_net.to_string(index=False))


In [ ]:
# 8c. chart_county_top_pairs — top 20 directed county->county pairs by IRS individuals
chart_county_top_pairs = run_sql('''
    SELECT origin_name, dest_name, origin_income_class, dest_income_class,
           irs_individuals_out AS people
    FROM county_migration_flows
    WHERE irs_individuals_out IS NOT NULL
    ORDER BY people DESC
    LIMIT 20
''', con)
save_processed(chart_county_top_pairs, cfg, 'chart_county_top_pairs.parquet')
load_to_duckdb(chart_county_top_pairs, 'chart_county_top_pairs', con)
register('chart_county_top_pairs', 'IRS SOI county migration (prepared)', IRS_URL,
         'Top 20 directed county->county pairs by IRS individuals (2023), with origin/dest income class.',
         len(chart_county_top_pairs))
print(f'chart_county_top_pairs: {len(chart_county_top_pairs)} rows')
print(chart_county_top_pairs.head(3).to_string(index=False))

### `chart_la_destinations` — Los Angeles County outbound detail (2023)
One row per **destination county** that LA County (FIPS `06037`) residents moved to in 2023, built
from `county_migration_flows WHERE origin_fips='06037'`. Columns: `dest_label` (county + state in
parens, same `LEFT(dest_fips,2)` -> `state_ref` crosswalk as `chart_county_net`), `dest_income_class`,
`people` (`irs_individuals_out`), `agi_millions` (`irs_agi_out/1e3`, since IRS AGI is in thousands),
and `agi_per_person` (`irs_agi_out*1000/individuals`, dollars per person). **All destination rows kept.**
Source for the LA two-panel detail chart (Chart C). Caveats: **2023 only**; identified county edges
cover ~60% of moves; **income class = place, not mover**.


In [ ]:
# 8c-bis. chart_la_destinations — LA County (06037) outbound detail, one row per destination
chart_la_destinations = run_sql('''
    SELECT
        dest_name,
        dest_name || ' (' || sr.abbrev || ')' AS dest_label,
        dest_income_class,
        irs_individuals_out AS people,
        ROUND(irs_agi_out / 1e3, 1) AS agi_millions,
        ROUND(irs_agi_out * 1000.0 / NULLIF(irs_individuals_out, 0), 0) AS agi_per_person
    FROM county_migration_flows f
    LEFT JOIN state_ref sr ON sr.fips = LEFT(f.dest_fips, 2)
    WHERE f.origin_fips = '06037'
    ORDER BY people DESC NULLS LAST
''', con)
save_processed(chart_la_destinations, cfg, 'chart_la_destinations.parquet')
load_to_duckdb(chart_la_destinations, 'chart_la_destinations', con)
register('chart_la_destinations', 'IRS SOI county migration (prepared)', IRS_URL,
         'LA County (FIPS 06037) outbound detail, 2023: one row per destination county with '
         'dest_label (county + state), dest_income_class, people (irs_individuals_out), '
         'agi_millions (irs_agi_out/1e3), and agi_per_person (irs_agi_out*1000/individuals). '
         'All destination rows kept. Source for the LA two-panel detail chart.',
         len(chart_la_destinations))
print(f'chart_la_destinations: {len(chart_la_destinations):,} rows')
print(chart_la_destinations.head(7).to_string(index=False))


In [ ]:
# 8d. Quality assertions on the prepared county tables
icn = run_sql('SELECT * FROM chart_county_income_class_net', con).set_index('income_class')
# Honest IN - OUT convention: below-median counties gained BOTH people and dollars.
assert int(icn.loc['below_median', 'net_people']) == 59420, icn
assert round(float(icn.loc['below_median', 'net_agi_billions']), 3) == 1.309, icn
# above-median is the mirror (net outflow-to-below of both people and dollars)
assert int(icn.loc['above_median', 'net_people']) == -59420, icn
assert round(float(icn.loc['above_median', 'net_agi_billions']), 3) == -1.309, icn
cnt = run_sql('SELECT COUNT(*) n FROM chart_county_net', con)['n'][0]
assert int(cnt) == 3143, cnt
pairs = run_sql('SELECT COUNT(*) n FROM chart_county_top_pairs', con)['n'][0]
assert int(pairs) == 20, pairs
# label present on every county row + spot-check LA
lbl = run_sql('SELECT COUNT(*) n FROM chart_county_net WHERE label IS NULL', con)['n'][0]
assert int(lbl) == 0, f'{lbl} chart_county_net rows missing label'
la = run_sql("SELECT label FROM chart_county_net WHERE name='Los Angeles County'", con)['label'][0]
assert la == 'Los Angeles County (CA)', la
# LA destinations table built and non-empty
lad = run_sql('SELECT COUNT(*) n FROM chart_la_destinations', con)['n'][0]
assert int(lad) > 0, lad
# LA destinations with >=500 movers (the Chart C right-panel floor) exist and lead with real metros
la_floor = run_sql('''
    SELECT dest_name FROM chart_la_destinations
    WHERE people >= 500 ORDER BY agi_per_person DESC LIMIT 10
''', con)['dest_name'].tolist()
assert 'New York County' in la_floor, la_floor
assert 'Miami-Dade County' in la_floor, la_floor
assert not any(c in la_floor for c in ['Pitkin County','Teton County','Summit County','Blaine County']), la_floor
print(f'county chart-ready tables OK: income-class net (honest in-out) + 3143-county net + 20 top pairs '
      f'+ label + {int(lad)} LA destinations; Chart C >=500 floor leads with metros ({la_floor[:3]})')


## 6. Package export (CSV + Excel + Parquet + codebook)

In [ ]:
codebook = {
    'year': 'Migration year = the later of the two years compared (IRS 2022–23 pair and ACS 2023 survey both map to 2023).',
    'origin_fips': 'FIPS code of the origin (state people moved FROM). Non-state codes: 72=Puerto Rico, 57=Foreign Country; U.S. Island Areas uses sentinel US-ISL.',
    'origin_state': 'Two-letter abbreviation of the origin.',
    'origin_state_name': 'Full name of the origin.',
    'origin_type': 'Origin location type: state, dc, territory (PR / U.S. Island Areas), or foreign.',
    'dest_fips': 'FIPS code of the destination (state people moved TO). Same coding as origin_fips.',
    'dest_state': 'Two-letter abbreviation of the destination.',
    'dest_state_name': 'Full name of the destination.',
    'dest_type': 'Destination location type: state, dc, territory, or foreign.',
    'irs_returns_out': 'IRS: number of tax returns (~households) for this origin→dest move, as reported in the ORIGIN state outflow file. Raw counts (not thousands).',
    'irs_returns_in': 'IRS: number of tax returns (~households) for this origin→dest move, as reported in the DESTINATION state inflow file. Usually equals irs_returns_out; differs slightly in later series.',
    'irs_individuals_out': 'IRS: number of personal exemptions (~people) for this move, from the origin outflow file.',
    'irs_individuals_in': 'IRS: number of personal exemptions (~people) for this move, from the destination inflow file.',
    'irs_agi_out': 'IRS: total adjusted gross income for this move (in THOUSANDS of dollars), from the origin outflow file.',
    'irs_agi_in': 'IRS: total adjusted gross income for this move (in THOUSANDS of dollars), from the destination inflow file.',
    'acs_migrants': 'Census ACS: estimated number of people who made this origin→dest move (survey estimate).',
    'acs_moe': 'Census ACS: margin of error (± people) on acs_migrants at 90% confidence.',
    'acs_moe_pct': 'acs_moe as a fraction of acs_migrants. Higher = less reliable.',
    'acs_reliable': 'True when acs_moe is less than 50% of acs_migrants (a usability cutoff for small survey flows).',
}

notes = '''
Sources:
  - IRS Statistics of Income (SOI) State-to-State Migration Data — https://www.irs.gov/statistics/soi-tax-stats-migration-data (public domain)
  - U.S. Census Bureau, American Community Survey (ACS) State-to-State Migration Flows — https://www.census.gov/data/tables/time-series/demo/geographic-mobility/state-to-state-migration.html (public domain)

Grain: one row per origin x destination x year (directed edge). True state-to-state moves only;
IRS summary rows (FIPS 96/97/98), overseas (59), and non-migrant (stayed-put) rows are excluded
(non-migrants live in a separate nonmigrants table).

Coverage: IRS 2012–2023 (year = second year of the pair); ACS 2011–2019, 2021–2024 (no 2020, ACS 1-year
suspended for COVID). Rows carry nulls where only one source covers that year.

Caveats:
  - IRS counts tax filers, not total population; ACS counts all residents 1 year and older.
  - IRS AGI is in THOUSANDS of dollars.
  - ACS flows are survey estimates; use acs_reliable / acs_moe_pct before headlining small numbers.
  - 2022 ACS Connecticut data has a known processing error (corrected in 2023).
'''

written = package_dataset(flows, cfg, name='state_migration_flows_v1',
                          codebook=codebook, notes=notes)
written

---
**Next:** `05-analysis.ipynb` for net-migration leaders/losers, income-weighted flows, and tax correlations; `04-viz.ipynb` for exploratory charts.

---
## Cleanup
Close the DuckDB connection so the write lock is released for other tools (DBCode, other notebooks).

In [ ]:
con.close()
print('connection closed')